# Tutorial: Regularization

> **Type:** Tutorial. Follow along step by step. Your only job is to make it work.  
> **Time:** ~1.5 hours  
> **Prerequisites:** Modules 13 (cross-validation), 16 (model evaluation)  
> **What you'll build:** A visual demonstration of overfitting, plus all three regularisation methods (Ridge, Lasso, Elastic Net) applied to real data, with regularisation paths showing how the penalty reshapes the model.

---

## What We're Building

We'll first *create* overfitting on purpose, so you can see exactly what it looks like. Then we'll fix it with L1, L2, and Elastic Net regularisation, and plot the regularisation paths to see how each method constrains the model differently.

## Step 0: Setup

In [ ]:
# !pip install scikit-learn matplotlib numpy pandas

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet,
    RidgeCV, LassoCV, ElasticNetCV
)
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)
plt.rcParams['figure.figsize'] = (12, 5)
print('Setup complete.')

---
## Part 1: Creating and Visualising Overfitting

### Step 1: Generate a Simple 1D Dataset

In [ ]:
# True function: y = sin(x) + noise
n_samples = 30
X_1d = np.sort(np.random.uniform(-3, 3, n_samples))
y_1d = np.sin(X_1d) + np.random.normal(0, 0.3, n_samples)
X_1d_test = np.linspace(-3, 3, 200)

plt.figure(figsize=(8, 4))
plt.scatter(X_1d, y_1d, color='#4E79A7', s=50, zorder=3, label='Training data')
plt.plot(X_1d_test, np.sin(X_1d_test), 'k--', linewidth=1.5, label='True function sin(x)')
plt.xlabel('x'); plt.ylabel('y')
plt.title('The Dataset (30 noisy samples of sin(x))', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

**Chart description.** The dataset: 30 noisy points scattered around the dashed curve y = sin(x), for x from −3 to 3. The points follow the wave from about −1.5 up to about 1.

### Step 2: Fit Polynomials of Increasing Degree

In [ ]:
from sklearn.model_selection import KFold

degrees = [1, 5, 15]
colours = ['#F28E2B', '#59A14F', '#E15759']
X_col   = X_1d.reshape(-1, 1)

# X_1d is sorted, so unshuffled folds would each test on an unseen x-range
# (pure extrapolation). Shuffle the folds so CV measures interpolation error.
cv_split = KFold(n_splits=5, shuffle=True, random_state=SEED)

train_errors, test_errors = [], []

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, deg, colour in zip(axes, degrees, colours):
    # Scale after expanding: x**15 reaches ~1e7, which breaks the least-squares fit
    pipe = make_pipeline(PolynomialFeatures(deg), StandardScaler(), LinearRegression())
    pipe.fit(X_col, y_1d)
    
    X_test_col = X_1d_test.reshape(-1, 1)
    y_fit  = pipe.predict(X_test_col)
    
    # Train error
    train_pred  = pipe.predict(X_col)
    train_rmse  = np.sqrt(mean_squared_error(y_1d, train_pred))
    train_errors.append(train_rmse)
    
    # Approximate test error via CV
    cv_rmse = np.sqrt(-cross_val_score(pipe, X_col, y_1d, cv=cv_split,
                                        scoring='neg_mean_squared_error').mean())
    test_errors.append(cv_rmse)
    
    ax.scatter(X_1d, y_1d, color='#4E79A7', s=30, zorder=3, label='Data')
    ax.plot(X_1d_test, np.sin(X_1d_test), 'k--', linewidth=1, label='True fn')
    ax.plot(X_1d_test, np.clip(y_fit, -3, 3), color=colour, linewidth=2, label=f'Degree {deg}')
    ax.set_title(f'Degree {deg}\nTrain RMSE={train_rmse:.3f}  CV RMSE={cv_rmse:.3f}',
                 fontweight='bold')
    ax.set_ylim(-2.5, 2.5)
    ax.legend(fontsize=8)

plt.suptitle('Underfitting → Good Fit → Overfitting', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('polynomial_overfitting.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSummary:")
for deg, tr, te in zip(degrees, train_errors, test_errors):
    print(f"  Degree {deg:2d}: Train RMSE={tr:.3f}  CV RMSE={te:.3f}")

**Chart description.** Three polynomial fits to the same 30 points, with the true sin(x) dashed. Degree 1 is a straight line that can't follow the wave: train RMSE 0.436, CV RMSE 0.458. Degree 5 follows the wave closely: train RMSE 0.250, CV RMSE 0.300. Degree 15 wiggles through the training points and swings wildly at both ends: train RMSE 0.200, but CV RMSE 35.963. That's underfitting, a good fit and overfitting.

> **What just happened?** Degree 1 underfits: a straight line can't follow sin(x), so both errors are high (train 0.44, CV 0.46). Degree 5 fits well (train 0.25, CV 0.30). Degree 15 has the lowest training error of the three (0.20) but a huge CV error (about 36): it memorised the noise and swings wildly between the training points. The gap between training and CV error is the signature of overfitting.
>
> Two details in the code matter. The folds are shuffled, because `X_1d` is sorted: unshuffled folds would each test on an x-range the model never saw, which measures extrapolation instead of fit. And the polynomial features are scaled, because `x**15` reaches about 10⁷, which breaks an unscaled least-squares fit.

---
## Part 2: Ridge Regression (L2)

### Step 3: Fix Overfitting with Ridge

In [ ]:
alphas = [0.001, 0.1, 1.0, 10.0, 100.0]

fig, axes = plt.subplots(1, len(alphas), figsize=(18, 4), sharey=True)

for ax, alpha in zip(axes, alphas):
    pipe = make_pipeline(PolynomialFeatures(15), StandardScaler(), Ridge(alpha=alpha))
    pipe.fit(X_col, y_1d)
    y_fit = pipe.predict(X_1d_test.reshape(-1, 1))
    cv_rmse = np.sqrt(-cross_val_score(pipe, X_col, y_1d, cv=cv_split,
                                        scoring='neg_mean_squared_error').mean())
    ax.scatter(X_1d, y_1d, color='#4E79A7', s=20, zorder=3)
    ax.plot(X_1d_test, np.sin(X_1d_test), 'k--', linewidth=1)
    ax.plot(X_1d_test, np.clip(y_fit, -2.5, 2.5), color='#E15759', linewidth=2)
    ax.set_title(f'α={alpha}\nCV RMSE={cv_rmse:.3f}', fontweight='bold')
    ax.set_ylim(-2.5, 2.5)

plt.suptitle('Ridge (L2) on Degree-15 Polynomial — Increasing α', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('ridge_regularisation.png', dpi=150, bbox_inches='tight')
plt.show()

**Chart description.** Five ridge fits of a degree-15 polynomial, with α rising from 0.001 to 100. At α = 0.001 the curve still wiggles, CV RMSE 0.602. At α = 0.1 it follows the wave smoothly, the best CV RMSE at 0.280. At α = 1.0 it's a little flatter, 0.349. At α = 10 and 100 it flattens too far and misses the wave: 0.495 and 0.673.

### Step 4: Regularisation Path for Ridge

In [ ]:
from sklearn.linear_model import ridge_regression

# Transform features
poly = PolynomialFeatures(15)
scaler = StandardScaler()
X_poly = scaler.fit_transform(poly.fit_transform(X_col))

alphas_path = np.logspace(-3, 4, 100)
coef_paths = []

for alpha in alphas_path:
    ridge = Ridge(alpha=alpha, fit_intercept=False)
    ridge.fit(X_poly, y_1d)
    coef_paths.append(ridge.coef_)

coef_paths = np.array(coef_paths)

plt.figure(figsize=(10, 5))
for i in range(coef_paths.shape[1]):
    plt.plot(np.log10(alphas_path), coef_paths[:, i], linewidth=1, alpha=0.6)
plt.xlabel('log₁₀(α)  →  increasing regularisation')
plt.ylabel('Coefficient value')
plt.title('Ridge Regularisation Path\n(all 16 polynomial coefficients)', fontweight='bold')
plt.axvline(0, color='grey', linestyle='--', linewidth=1)
plt.tight_layout()
plt.savefig('ridge_path.png', dpi=150, bbox_inches='tight')
plt.show()

print("As α → ∞, ALL coefficients shrink toward zero.")
print("Ridge keeps all features but makes them small.")

**Chart description.** The ridge regularisation path: each line is one of the 16 polynomial coefficients, plotted against log₁₀(α) from −3 to 4. At small α the coefficients spread between about −1.8 and 1.9. As α grows they all shrink smoothly toward 0, and by log₁₀(α) = 3 they're all close to it. None reaches exactly 0 earlier than the others.

---
## Part 3: Lasso Regression (L1)

### Step 5: Lasso Drives Coefficients to Exactly Zero

In [ ]:
alphas_lasso = np.logspace(-4, 0, 100)
lasso_coef_paths = []

for alpha in alphas_lasso:
    lasso = Lasso(alpha=alpha, max_iter=10000, fit_intercept=False)
    lasso.fit(X_poly, y_1d)
    lasso_coef_paths.append(lasso.coef_)

lasso_coef_paths = np.array(lasso_coef_paths)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Lasso path
for i in range(lasso_coef_paths.shape[1]):
    axes[0].plot(np.log10(alphas_lasso), lasso_coef_paths[:, i], linewidth=1, alpha=0.7)
axes[0].set_xlabel('log₁₀(α)  →  increasing regularisation')
axes[0].set_ylabel('Coefficient value')
axes[0].set_title('Lasso (L1) Regularisation Path', fontweight='bold')

# Count non-zero coefficients
n_nonzero = (np.abs(lasso_coef_paths) > 1e-4).sum(axis=1)
axes[1].plot(np.log10(alphas_lasso), n_nonzero, color='#E15759', linewidth=2)
axes[1].set_xlabel('log₁₀(α)')
axes[1].set_ylabel('Number of non-zero coefficients')
axes[1].set_title('Lasso Feature Selection Effect\n(features driven to zero)', fontweight='bold')

plt.tight_layout()
plt.savefig('lasso_path.png', dpi=150, bbox_inches='tight')
plt.show()

print("Lasso drives coefficients to EXACTLY zero — this is feature selection.")
print("Ridge shrinks them but never zeros them out.")

**Chart description.** Left: the lasso regularisation path. Coefficients shrink as α grows, and many hit exactly 0 and stay there, unlike ridge. One large coefficient survives until log₁₀(α) is near 0. Right: the number of non-zero coefficients against log₁₀(α). It falls in steps, with some small jumps back up, from 11 at the smallest α to 0 at the largest.

---
## Part 4: Apply to Real Data and Compare All Three

### Step 6: Breast Cancer Dataset — Logistic with L1/L2/ElasticNet

In [ ]:
# Use breast cancer dataset: 30 features
bc = load_breast_cancer()
X_bc, y_bc = bc.data, bc.target
feature_names = bc.feature_names

scaler_bc = StandardScaler()
X_bc_s = scaler_bc.fit_transform(X_bc)

# Cross-validated selection of best alpha
ridge_cv    = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=5)
lasso_cv    = LassoCV(alphas=np.logspace(-4, 1, 100), cv=5, max_iter=10000)
enet_cv     = ElasticNetCV(alphas=np.logspace(-4, 1, 50), l1_ratio=[0.1, 0.5, 0.9], cv=5)

# Note: RidgeCV is a regressor, so we use it on a regression-like task
# For fair comparison, use cross_val_score on classification metric
from sklearn.linear_model import LogisticRegressionCV

results = {}
for penalty, solver in [('l2', 'lbfgs'), ('l1', 'saga'), ('elasticnet', 'saga')]:
    kwargs = {'penalty': penalty, 'solver': solver, 'max_iter': 10000,
              'Cs': 20, 'cv': 5, 'random_state': SEED}
    if penalty == 'elasticnet':
        kwargs['l1_ratios'] = [0.1, 0.5, 0.9]
    model = LogisticRegressionCV(**kwargs)
    model.fit(X_bc_s, y_bc)
    scores = cross_val_score(model, X_bc_s, y_bc, cv=5, scoring='roc_auc')
    n_nonzero = (np.abs(model.coef_[0]) > 1e-4).sum()
    results[penalty] = {
        'ROC-AUC': scores.mean(),
        'std': scores.std(),
        'features_used': n_nonzero,
        'best_C': model.C_[0],
    }

print(f"{'Penalty':15s} {'ROC-AUC':>10} {'±':>6} {'Features used':>15} {'Best C':>10}")
print("-" * 60)
for penalty, r in results.items():
    print(f"{penalty:15s} {r['ROC-AUC']:>10.4f} {r['std']:>6.4f} {r['features_used']:>15} {r['best_C']:>10.4f}")

### Step 7: Visualise Which Features Lasso Zeroes Out

In [ ]:
# Refit Lasso logistic regression and inspect coefficients
from sklearn.linear_model import LogisticRegression

# Use best C from lasso_cv above
lasso_model = LogisticRegressionCV(
    penalty='l1', solver='saga', Cs=30, cv=5, max_iter=10000, random_state=SEED
)
lasso_model.fit(X_bc_s, y_bc)
coefs = lasso_model.coef_[0]

# Sort by absolute value
idx   = np.argsort(np.abs(coefs))[::-1]
coefs_sorted  = coefs[idx]
names_sorted  = feature_names[idx]
colours = ['#E15759' if c != 0 else '#CCCCCC' for c in coefs_sorted]

plt.figure(figsize=(14, 5))
plt.bar(range(len(coefs_sorted)), coefs_sorted, color=colours)
plt.xticks(range(len(names_sorted)), names_sorted, rotation=90, fontsize=8)
plt.axhline(0, color='black', linewidth=1)
plt.title(f'Lasso Coefficients (grey = zeroed out, {(coefs==0).sum()} features eliminated)',
          fontweight='bold')
plt.ylabel('Coefficient value')
plt.tight_layout()
plt.savefig('lasso_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

**Chart description.** Lasso logistic regression coefficients for the 30 breast cancer features, sorted by size. Red bars are kept and grey are zeroed out. 13 features are eliminated and 17 kept. The largest are negative: worst radius at about −3.3 and radius error at about −2.2. Compactness error has the largest positive coefficient, about 0.6.

> **What just happened?** The grey bars are the features Lasso eliminated by setting their coefficients exactly to zero. Ridge would have kept every feature with small non-zero values. That's the fundamental difference: Lasso selects; Ridge shrinks.

---
## Part 5: Diagnosing with Learning Curves

### Step 8: Would More Data Help, or Does the Model Need to Change?

In [ ]:
from sklearn.model_selection import learning_curve

# Plain logistic regression on the breast cancer data from Part 4 —
# would more training data help this model, or does it need to change?
learning_model = LogisticRegression(max_iter=1000, random_state=SEED)

train_sizes, train_scores, val_scores = learning_curve(
    learning_model, X_bc_s, y_bc,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5, scoring='accuracy',
)

train_mean = train_scores.mean(axis=1)
val_mean = val_scores.mean(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, marker='o', color='#4E79A7', label='Training accuracy')
plt.plot(train_sizes, val_mean, marker='o', color='#E15759', label='Validation accuracy')
plt.xlabel('Training set size')
plt.ylabel('Accuracy')
plt.title('Learning Curve: Logistic Regression on Breast Cancer Data', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Training accuracy range: {train_mean.min():.3f} - {train_mean.max():.3f}")
print(f"Validation accuracy range: {val_mean.min():.3f} - {val_mean.max():.3f}")
print(f"Final gap (train - val): {train_mean[-1] - val_mean[-1]:.3f}")

**Chart description.** A learning curve for logistic regression on the breast cancer data. Accuracy against training set size, from about 45 to 455 rows. Training accuracy stays between 0.98 and 1.0. Validation accuracy starts at about 0.77 with the smallest set, jumps to about 0.95 at 90 rows and climbs slowly to about 0.98. The gap closes as data grows, so more data still helps a little.

> **What just happened?** Training accuracy stays near-perfect at every training-set size (0.98-1.00). Validation accuracy starts much lower (0.775 with the smallest slice of training data) and climbs steadily as more data is added, until the gap nearly closes (0.009 at full size). That's the high-variance signature at small sample sizes, and it's the case where more data helps. The model's assumptions weren't wrong; it just didn't have enough rows to pin them down. If both curves had converged to a similarly *low* accuracy instead, that would point to high bias. No amount of extra data would fix that, because the model itself would be too simple for the signal.

---
## Part 6: Reproducing the Book's Worked Example

### Step 9: Degree-15 Polynomial on a Noisy Quadratic, With a Held-Out Test Set

Everything above estimated test error with cross-validation. The book's Worked Example uses a held-out test set instead: 60 points sampled from `y = 0.5x^2 - x + 2` plus Gaussian noise, split 70/30, then a degree-15 polynomial fit three ways: unregularised, `RidgeCV`, and `LassoCV`.

In [ ]:
rng_q = np.random.RandomState(SEED)
X_q = rng_q.uniform(-5, 5, 60)
y_q = 0.5 * X_q**2 - X_q + 2 + rng_q.normal(0, 2, 60)
X_q = X_q.reshape(-1, 1)

X_train_q, X_test_q, y_train_q, y_test_q = train_test_split(
    X_q, y_q, test_size=0.3, random_state=SEED
)

poly15 = PolynomialFeatures(degree=15, include_bias=False)
scaler_q = StandardScaler()
X_train_poly = scaler_q.fit_transform(poly15.fit_transform(X_train_q))
X_test_poly = scaler_q.transform(poly15.transform(X_test_q))

# Unregularised
lr_q = LinearRegression().fit(X_train_poly, y_train_q)
train_rmse_unreg = np.sqrt(mean_squared_error(y_train_q, lr_q.predict(X_train_poly)))
test_rmse_unreg = np.sqrt(mean_squared_error(y_test_q, lr_q.predict(X_test_poly)))
print(f"Unregularised:  train RMSE {train_rmse_unreg:.3f}  test RMSE {test_rmse_unreg:.3f}")

# RidgeCV
ridge_q = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=5)
ridge_q.fit(X_train_poly, y_train_q)
train_rmse_ridge = np.sqrt(mean_squared_error(y_train_q, ridge_q.predict(X_train_poly)))
test_rmse_ridge = np.sqrt(mean_squared_error(y_test_q, ridge_q.predict(X_test_poly)))
print(f"Ridge (alpha={ridge_q.alpha_:.3f}):  train RMSE {train_rmse_ridge:.3f}  test RMSE {test_rmse_ridge:.3f}")

# LassoCV
lasso_q = LassoCV(alphas=np.logspace(-4, 1, 100), cv=5, max_iter=10000)
lasso_q.fit(X_train_poly, y_train_q)
train_rmse_lasso = np.sqrt(mean_squared_error(y_train_q, lasso_q.predict(X_train_poly)))
test_rmse_lasso = np.sqrt(mean_squared_error(y_test_q, lasso_q.predict(X_test_poly)))
n_zero = int((lasso_q.coef_ == 0).sum())
n_total = len(lasso_q.coef_)
print(f"Lasso (alpha={lasso_q.alpha_:.3f}):  train RMSE {train_rmse_lasso:.3f}  test RMSE {test_rmse_lasso:.3f}  ({n_zero} of {n_total} coefficients zeroed)")

---
## What You Built

A complete visual tour of regularisation. You created overfitting from scratch with polynomial regression, and fixed it with Ridge, watching every coefficient shrink. You compared that with Lasso, which eliminates features entirely. You applied all three methods to real classification data with cross-validated alpha selection. And you used a learning curve to diagnose whether more data or more regularisation is the right next move.

## What's Next

- Read `explanation.md` for the bias-variance trade-off and the geometry of L1 vs. L2.
- Use `howto.md` when you need to apply regularisation to your own models.
- Keep `reference.md` open for the method comparison table.
- **Extension challenge:** use `ElasticNetCV` to find the best `l1_ratio` on the breast cancer data. Try values from 0.1 to 0.9. Which `l1_ratio` gives the best CV score? Does it keep more or fewer features than pure L1?

---

**Next module:** `19_unsupervised_learning`, which shifts from labelled to unlabelled data: clustering and anomaly detection.